In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql. window import Window

In [0]:
df=spark.read.format('parquet')\
    .load('abfss://bronze@projecte2e.dfs.core.windows.net/customers')

In [0]:
df.display()

In [0]:
df=df.drop('_rescued_data')
df.display()

### Data Transformation using split(****)

In [0]:
df=df.withColumn('Domain',split(col('email'),'@')[1])
df.display()

In [0]:
df.groupBy('Domain').agg(count('customer_id').alias('total_customers')).sort('total_customers',ascending=False).display()

In [0]:
df_gmail=df.filter(col("domain")=='gmail.com')
df_gmail.display()

In [0]:
df=df.withColumn('Full_name',concat(col('first_name'),lit(' '),col('last_name')))
df=df.drop("first_name",'last_name')
df.display()

In [0]:
df.write.mode('overwrite')\
    .format("delta")\
        .save('abfss://silver@projecte2e.dfs.core.windows.net/customers')

# creating a table in databricks

In [0]:
%sql
create schema project_cata.silver

In [0]:
%sql
create table if not exists project_cata.silver.customers_silver
using delta
location "abfss://silver@projecte2e.dfs.core.windows.net/customers"

In [0]:
%sql
select * from project_cata.silver.customers_silver